# Data Retrieval MediaCloud

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | MediaCloud API — `mediacloud` Python client |
| **Method** | `story_count_over_time` for daily counts · `story_list` for full story metadata |
| **Collection** | 34412234 — U.S. Top Online News (187 outlets) |
| **Queries** | `trump`, `harris`, `election` |
| **Volume** | **235,191** story records · **123** daily aggregate rows |
| **Saved to** | `Data/1_Bronze/Newspapers/mediacloud_daily.csv` · `mediacloud_stories.csv` · `mediacloud_features.csv` |

### mediacloud_daily.csv — 123 rows × 18 columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Calendar date |
| `trump`, `harris`, `election` | int | Raw daily story count per query |
| `trump_pct`, `harris_pct`, `election_pct` | float | Share of total daily stories (%) |
| `trump_share` | float | Trump stories as % of Trump + Harris |
| `total_coverage` | int | Sum of all three query counts |
| `log_trump`, `log_harris`, `log_election`, `log_total_coverage` | float | log1p-transformed counts |
| `trump_7d`, `harris_7d`, `election_7d`, `trump_share_7d`, `total_coverage_7d` | float | 7-day rolling averages |

### mediacloud_stories.csv — 235,191 rows × 6 columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Publication date |
| `query` | str | Search query used to retrieve the story (`trump` / `harris` / `election`) |
| `title` | str | Article headline |
| `url` | str | Full article URL |
| `source` | str | Media outlet name |
| `pub_date` | datetime | Full publication datetime |


<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Daily Story Counts](#1-daily-story-counts)
  - [mediacloud_daily.csv columns](#mediacloud_dailycsv-columns)
  - [mediacloud_features.csv columns](#mediacloud_featurescsv-columns)
- [2. Full Story Metadata](#2-full-story-metadata)
  - [mediacloud_stories.csv columns](#mediacloud_storiescsv-columns)


## Setup

In [ ]:
import datetime as dt
import os
import time
import pandas as pd
import numpy as np
import mediacloud.api
from pathlib import Path

# API key
MC_API_KEY = os.getenv("MC_API_KEY", "19e0165245bd1dfe3772cc9f65d3085a5614bd95")

# Date range
START_DATE = dt.date(2024, 7, 5)
END_DATE   = dt.date(2024, 11, 4)

# U.S. Top Online News collection
US_COLLECTION = [34412234]

# Search queries
QUERIES = {
    "trump":    '"Trump" AND (election OR president OR campaign OR vote)',
    "harris":   '"Harris" AND (election OR president OR campaign OR vote)',
    "election": '"presidential election" AND 2024',
}

# Output paths (Bronze layer)
_root = Path(os.getcwd())
for _ in range(5):
    if (_root / "Data").exists():
        break
    _root = _root.parent
BRONZE = _root / "Data" / "1_Bronze" / "Newspapers"
BRONZE.mkdir(parents=True, exist_ok=True)

mc = mediacloud.api.SearchApi(MC_API_KEY)
print(f"Period : {START_DATE} -> {END_DATE}  ({(END_DATE - START_DATE).days + 1} days)")
print(f"Output : {BRONZE}")


## 1. Daily Story Counts

### mediacloud_daily.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Calendar date |
| `trump`, `harris`, `election` | int | Raw daily story count per query |
| `trump_pct`, `harris_pct`, `election_pct` | float | Share of total daily stories (%) |
| `trump_share` | float | Trump stories as % of Trump + Harris |
| `total_coverage` | int | Sum of all three query counts |
| `log_trump`, `log_harris`, `log_election`, `log_total_coverage` | float | log1p-transformed counts |
| `trump_7d`, `harris_7d`, `election_7d`, `trump_share_7d`, `total_coverage_7d` | float | 7-day rolling averages |

### mediacloud_features.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Calendar date |
| `trump`, `harris`, `election` | int | Raw daily story counts |
| `trump_share` | float | Trump stories as % of Trump + Harris |
| `total_coverage` | int | Sum of all three query counts |
| `log_trump`, `log_harris`, `log_total_coverage` | float | log1p-transformed counts |
| `trump_7d`, `harris_7d`, `trump_share_7d` | float | 7-day rolling averages |

In [ ]:
# Fetch daily story counts for each query and compute derived features.

def fetch_daily_counts(query: str, label: str) -> pd.DataFrame:
    print(f"  Fetching '{label}' ...", end=" ", flush=True)
    results = mc.story_count_over_time(
        query,
        START_DATE,
        END_DATE,
        collection_ids=US_COLLECTION,
    )
    df = pd.DataFrame(results)
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df = df.rename(columns={"count": label, "ratio": f"{label}_pct"})
    df = df[["date", label, f"{label}_pct"]]
    print(f"{len(df)} rows")
    time.sleep(1)
    return df

print("Fetching daily counts ...")
dfs = [fetch_daily_counts(q, lbl) for lbl, q in QUERIES.items()]

daily = dfs[0]
for df in dfs[1:]:
    daily = daily.merge(df, on="date", how="outer")
daily = daily.sort_values("date").reset_index(drop=True)
daily = daily[(daily["date"] >= START_DATE) & (daily["date"] <= END_DATE)].copy()

# Derived features
daily["trump_share"]    = daily["trump"] / (daily["trump"] + daily["harris"]).replace(0, np.nan)
daily["total_coverage"] = daily["trump"] + daily["harris"] + daily["election"]
for col in ["trump", "harris", "election", "total_coverage"]:
    daily[f"log_{col}"] = np.log1p(daily[col])
for col in ["trump", "harris", "election", "trump_share", "total_coverage"]:
    daily[f"{col}_7d"] = daily[col].rolling(7, min_periods=1).mean()

# Save full daily table
daily.to_csv(BRONZE / "mediacloud_daily.csv", index=False)
print(f"Saved: mediacloud_daily.csv  ({daily.shape[0]} rows x {daily.shape[1]} cols)")

# Save compact feature table for predictive model
CORE_COLS = [
    "date", "trump", "harris", "election",
    "trump_share", "total_coverage",
    "log_trump", "log_harris", "log_total_coverage",
    "trump_7d", "harris_7d", "trump_share_7d",
]
daily[CORE_COLS].to_csv(BRONZE / "mediacloud_features.csv", index=False)
print(f"Saved: mediacloud_features.csv  ({len(CORE_COLS)} cols)")


## 2. Full Story Metadata

### mediacloud_stories.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Publication date |
| `query` | str | Search query used to retrieve the story (`trump` / `harris` / `election`) |
| `title` | str | Article headline |
| `url` | str | Full article URL |
| `source` | str | Media outlet name |
| `pub_date` | datetime | Full publication datetime |

In [ ]:
# Fetch full story metadata (title, url, source) for all days and queries.
# Pagination with retry + exponential backoff.
# After the main pass, any dates still missing are re-fetched automatically.

MAX_PAGES          = 3   # ~100 stories per page
SLEEP_BETWEEN_Q    = 5   # seconds between queries within a day
SLEEP_BETWEEN_DAYS = 10  # seconds between days


def fetch_stories_for_day(day_date, query, max_pages=MAX_PAGES, max_retries=4):
    # Return a list of raw story dicts for one day + query.
    pagination_token = None
    stories = []
    for page in range(max_pages):
        for attempt in range(max_retries):
            try:
                results, pagination_token = mc.story_list(
                    query,
                    day_date,
                    day_date,
                    collection_ids=US_COLLECTION,
                    pagination_token=pagination_token,
                )
                stories.extend(results)
                break
            except Exception as e:
                wait = 5 * (2 ** attempt)
                print(f"    [{e.__class__.__name__}] retrying in {wait}s ...", flush=True)
                time.sleep(wait)
        else:
            print(f"    Skipping {day_date} / page {page} after {max_retries} attempts.")
            return stories
        if not pagination_token or not results:
            break
        time.sleep(2)
    return stories


def collect_stories(date_range):
    # Fetch stories for every date in date_range. Returns list of row dicts.
    records = []
    for day in date_range:
        day_date = day.date() if hasattr(day, "date") else day
        for label, query in QUERIES.items():
            for s in fetch_stories_for_day(day_date, query):
                records.append({
                    "date":     day_date,
                    "query":    label,
                    "title":    s.get("title", ""),
                    "url":      s.get("url", ""),
                    "source":   s.get("media_name", ""),
                    "pub_date": s.get("publish_date", ""),
                })
            time.sleep(SLEEP_BETWEEN_Q)
        day_count = sum(1 for r in records if r["date"] == day_date)
        print(f"{day_date}: {day_count} stories fetched so far", flush=True)
        time.sleep(SLEEP_BETWEEN_DAYS)
    return records


# Main fetch pass
print("Fetching stories (this takes ~30 min) ...")
all_records = collect_stories(pd.date_range(START_DATE, END_DATE, freq="D"))
stories_df  = pd.DataFrame(all_records)

# Re-fetch any dates skipped due to API errors
full_range  = set(pd.date_range(START_DATE, END_DATE, freq="D").date)
found_dates = set(pd.to_datetime(stories_df["date"]).dt.date)
missing     = sorted(full_range - found_dates)
if missing:
    print(f"Re-fetching {len(missing)} missing dates: {missing}")
    extra      = collect_stories(missing)
    stories_df = pd.concat([stories_df, pd.DataFrame(extra)], ignore_index=True)

# Deduplicate and sort
stories_df["date"] = pd.to_datetime(stories_df["date"])
stories_df = (
    stories_df
    .drop_duplicates(subset="url")
    .sort_values("date")
    .reset_index(drop=True)
)

stories_df.to_csv(BRONZE / "mediacloud_stories.csv", index=False)
print(f"Saved: mediacloud_stories.csv  ({len(stories_df):,} rows)")

# Verify all dates present
dates_present = stories_df["date"].dt.date.unique()
still_missing = sorted(full_range - set(dates_present))
if still_missing:
    print(f"WARNING: still missing {len(still_missing)} dates: {still_missing}")
else:
    print("All 123 days present.")
